# Init Lakehouse

In [1]:
%%configure -f
{
    "defaultLakehouse": {"name": "DE_LH_100_BondedWarehouse"}
}

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, -1, Finished, Available, Finished)

# Init Imports (these need cutting-down post creation)

In [2]:
import os
import csv
import re
import shutil
import unicodedata
import pandas as pd

import notebookutils

#from decimal import Decimal
from datetime import datetime
from datetime import timedelta
#from collections import Counter
#from functools import reduce
import time

#from pyspark import StorageLevel
from pyspark.sql import DataFrame, Row
from pyspark.sql.functions import col, lit, when, concat, concat_ws, coalesce, count, monotonically_increasing_id, sum, to_date, udf, current_timestamp, length, substring, split, size, asc, row_number, desc, trim, regexp_replace
from pyspark.sql.functions import broadcast, hash, array, expr, array_distinct, date_format
from pyspark.sql.types import *
#from pyspark.sql import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 3, Finished, Available, Finished)

# Init Export Process

In [3]:
def save_dataframe_to_csv(df, file_path, show_header=False, mode='overwrite'):
    """
    Save a DataFrame as a single CSV file in a PySpark application.

    Parameters:
    df (pyspark.sql.DataFrame): The DataFrame to save.
    file_path (str): The path to save the CSV file.
    header (bool): Whether to include the header in the CSV file. Default is True.
    mode (str): The write mode. Options are 'overwrite', 'append', 'ignore', 'error' or 'errorifexists'. Default is 'overwrite'.

    Returns:
    None
    """

    use_pipes = len(df.columns) != 1
    print(f'Add pipes: {use_pipes}')
    print(f'Show headers: {show_header}')

    pandas_df = df.toPandas()
    
    # Replace newlines and carriage returns
    pandas_df = pandas_df.replace({r'\r\n': ' ', r'\n': ' ', r'\r': ' '}, regex=True)

    # Create a string representation of the DataFrame with '|' as separator
    # Escape special characters such as commas and pipes
    if use_pipes:
        csv_data = pandas_df.to_csv(sep="|", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")
    else:
        csv_data = pandas_df.to_csv(sep="~", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")

    # Add trailing pipe '|' at the end of each line
    csv_data_with_pipe = '\n'.join([line + '|' for line in csv_data.split('\n') if line])

    # Write to the file
    with open(file_path, 'w') as f:
        f.write(csv_data_with_pipe)


StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 4, Finished, Available, Finished)

# Init Debug & Incremental Vars

In [4]:
workspace_name = notebookutils.mssparkutils.env.getWorkspaceName()

if "DEV" in workspace_name.upper():
    debug = True
    incremental_run = False
    default_days_lag: int = 7

elif "UAT" in workspace_name.upper():
    debug = True
    incremental_run = True
    default_days_lag: int = 7

else:
    debug = False
    incremental_run = True
    default_days_lag: int = 1

if debug:
    print(debug , incremental_run)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 5, Finished, Available, Finished)

True False


# Init Days Lag Var

In [5]:
filterdate_pipe = ''

#default_days_lag: int = 1

enable_string_truncation = True
create_hash_cols: bool = False
transfer_file: bool = False
retain_error_records_in_ouput_file: bool = False

# Override Debug

#debug = False   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#debug = True   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

# Override Full Run 

#incremental_run = False    #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#incremental_run = True     #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 6, Finished, Available, Finished)

In [6]:
filterdate = datetime.now() - timedelta(days=default_days_lag)
filterdate = filterdate.date()

if debug:
    print(f'Get shipments from: {filterdate}')

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 7, Finished, Available, Finished)

Get shipments from: 2025-05-02


# Init Query(s)

## Level 1 - Shipments

In [7]:
shipments_1_df = spark.sql(f"""
SELECT
'1' AS Shipment_Level_No,
whscontainertable.containerid AS Shipment_Reference,
CAST(whscontainertable.closecontainerutcdatetime AS DATE) AS Date_of_Shipment,
'EXPORT' AS Project_Key,
CASE WHEN salestable.salestype = 3 THEN "1" ELSE "2" END AS NOTC_A,
CASE WHEN salestable.salestype = 3 THEN "1" ELSE "9" END AS NOTC_B,
'' AS Container_ID,
'' AS Container_Seal,
'' AS Ship_To,
CASE WHEN transportidlookup.Shippingcarrierservice IS NULL THEN whsshipmenttable.hslbordertransportid ELSE transportidlookup.Bordertransportid END AS Border_Transport_ID,
CASE WHEN transportidlookup.Shippingcarrierservice IS NULL THEN whsshipmenttable.hslbordertransportnationality ELSE transportidlookup.Bordertransportnationality END AS Border_Transport_Nationality,
CASE WHEN transportidlookup.Shippingcarrierservice IS NULL THEN whsshipmenttable.hslinlandtransportid ELSE transportidlookup.Inlandtransportid END AS Inland_Transport_ID,
'' AS GB_Transport_Cost,
'' AS GB_Transport_Cost_Currency,
'' AS GB_Insurance_Cost,
'' AS GB_Insurance_Cost_Currency,
'' AS Incoterm_Code,
'' AS Conveyance_Reference,
'' AS Additional_Documents_Supporting_Document_Code_1,
'' AS Additional_Documents_Supporting_Document_Reference_1,
'' AS Additional_Documents_Complementary_Info_1,
'' AS Additional_Documents_Supporting_Document_Code_2,
'' AS Additional_Documents_Supporting_Document_Reference_2,
'' AS Additional_Documents_Complementary_Info_2,
'' AS Additional_Documents_Supporting_Document_Code_3,
'' AS Additional_Documents_Supporting_Document_Reference_3,
'' AS Additional_Documents_Complementary_Info_3,
'' AS Additional_Documents_Transit_Document_Code_1,
'' AS Additional_Documents_Transit_Document_Reference_1,
'' AS Additional_Documents_Transit_Document_Code_2,
'' AS Additional_Documents_Transit_Document_Reference_2,
'' AS Additional_Documents_Transit_Document_Code_3,
'' AS Additional_Documents_Transit_Document_Reference_3,
'' AS Additional_Documents_Additional_Reference_Code_1,
'' AS Additional_Documents_Additional_Reference_1,
'' AS Additional_Documents_Additional_Reference_Code_2,
'' AS Additional_Documents_Additional_Reference_2,
'' AS Additional_Documents_Additional_Reference_Code_3,
'' AS Additional_Documents_Additional_Reference_3,
'' AS Location_Name,
'' AS Valuation_Indicator,
'' AS Valuation_Method,
'' AS Estimated_Arrival_Date,
'' AS Estimated_Arrival_Time,
'' AS Additional_Documents_Code_1,
'' AS Additional_Documents_Reference_1,
'' AS Additional_Documents_Code_2,
'' AS Additional_Documents_Reference_2,
'' AS Additional_Documents_Code_3,
'' AS Additional_Documents_Reference_3,
'' AS Additional_Documents_Code_4,
'' AS Additional_Documents_Reference_4,
'' AS Additional_Documents_Code_5,
'' AS Additional_Documents_Reference_5,
'' AS Additional_Documents_Code_6,
'' AS Additional_Documents_Reference_6,
'' AS Commission_and_Brokerage_Value_AB,
'' AS Commission_and_Brokerage_Currency_AB,
'' AS Container_and_Packing_Value_AD,
'' AS Container_and_Packing_Currency_AD,
'' AS Royalties_and_Licensing_Fees_AI,
'' AS Royalties_and_Licensing_Fees_Currency_AI,
'' AS Freight_Charges_AK,
'' AS Freight_Charges_Currency_AK,
'' AS Insurance_Costs_AK,
'' AS Insurance_Costs_Currency_AK,
'' AS Other_Payments_AL,
'' AS Other_Payments_Currency_AL,
'' AS Internal_EU_Transport_Value_1X,
'' AS Internal_EU_Transport_Currency_1X,
'' AS Transport_After_Arrival_Value_BA,
'' AS Transport_After_Arrival_Currency_BA,
'' AS Import_Duties_Value_BC,
'' AS Import_Duties_Currency_BC,
'' AS Buying_Commission_Value_BF,
'' AS Buying_Commission_Currency_BF,
REPLACE(REPLACE(whsshipmenttable.hslmovementkey, ' ', ''),'-','_') AS Movement_Key,
'' AS AIS_Movement,
'' AS NCTS_Movement,
'' AS Declaration_Type,
CASE WHEN custtable.custgroup IN ("Interco", "3rd Party", "RTV") THEN "B2B" ELSE "B2C" END AS Transaction_Type,
'' AS Export_Goods_Location,
'' AS NCTS_Location_Type,
'' AS NCTS_Goods_Location,
'' AS AIS_Goods_Location,
'' AS Customs_Office_of_Departure,
'' AS Customs_Office_of_Transit,
'' AS Customs_Office_of_Transit_2,
'' AS Customs_Office_of_Transit_3,
'' AS Customs_Office_of_Transit_4,
'' AS Customs_Office_of_Transit_5,
'' AS Customs_Office_of_Transit_6,
'' AS Guarantee_Key,
'' AS Freight_Forwarder,
'' AS Consolidator,
'' AS Manufacturer,
'' AS Warehousekeeper,
'' AS Border_Transport_Type,
'' AS Border_Transport_Mode,
'' AS Inland_Transport_Mode,
'' AS Inland_Transport_Type,
'' AS Destination_Transport_Mode,
'' AS Carrier_Key,
logisticsaddresscountryregion.isocode AS Destination_Country,
'' AS Destination_Office,
'' AS Exit_Office,
'' AS Place_of_Loading,
'' AS Place_of_Unloading,
whscontainertable.shipmentid AS Key


FROM whscontainertable

INNER JOIN whsshipmenttable
ON  whscontainertable.shipmentid = whsshipmenttable.shipmentid
AND whscontainertable.dataareaid = whsshipmenttable.dataareaid

INNER JOIN salestable
ON whsshipmenttable.ordernum = salestable.salesid
AND whsshipmenttable.dataareaid = salestable.dataareaid

INNER JOIN custtable 
ON salestable.custaccount = custtable.accountnum
AND salestable.dataareaid = custtable.dataareaid

LEFT JOIN logisticspostaladdress
ON salestable.deliverypostaladdress = logisticspostaladdress.recid

LEFT JOIN logisticsaddresscountryregion
ON logisticspostaladdress.countryregionid = logisticsaddresscountryregion.countryregionid

LEFT JOIN transportidlookup
ON whsshipmenttable.carrierservicecode = transportidlookup.Shippingcarrierservice



WHERE whscontainertable.dataareaid IN ('end.', 'END.')
AND logisticspostaladdress.validfrom <= current_date()
AND logisticspostaladdress.validto > current_date()
AND whscontainertable.containerstatus = 2
AND logisticspostaladdress.countryregionid != 'GBR'
AND salestable.dlvmode NOT LIKE '%DPD%'
AND salestable.dlvmode NOT LIKE '%Royal%'
AND logisticspostaladdress.zipcode NOT LIKE 'BF%'

AND whscontainertable.closecontainerutcdatetime >= '{filterdate}'

""")
if debug:
      display(shipments_1_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 746d5960-ebf4-4452-837d-beb03c5c9c76)

In [8]:
shipments_1_df = shipments_1_df.select(
    substring(col("Shipment_Level_No").cast("string"),1, 8).alias("Shipment_Level_No"),
    substring(col("Shipment_Reference").cast("string"),1, 25).alias("Shipment_Reference"),
    col("Date_of_Shipment").cast("date").alias("Date_of_Shipment"),
    substring(col("Project_Key").cast("string"),1, 10).alias("Project_Key"),
    substring(col("NOTC_A").cast("string"),1, 1).alias("NOTC_A"),
    substring(col("NOTC_B").cast("string"),1, 1).alias("NOTC_B"),
    col("Container_ID").cast("string").alias("Container_ID"),
    col("Container_Seal").cast("string").alias("Container_Seal"),
    col("Ship_To").cast("string").alias("Ship_To"),
    substring(col("Border_Transport_ID").cast("string"),1, 35).alias("Border_Transport_ID"),
    substring(col("Border_Transport_Nationality").cast("string"),1, 2).alias("Border_Transport_Nationality"),

    when(substring(col("Inland_Transport_ID").cast("string"),1, 27) == "N/A", "N A").otherwise(col("Inland_Transport_ID")).alias("Inland_Transport_ID"),

    col("GB_Transport_Cost").cast("string").alias("GB_Transport_Cost"),
    col("GB_Transport_Cost_Currency").cast("string").alias("GB_Transport_Cost_Currency"),
    col("GB_Insurance_Cost").cast("string").alias("GB_Insurance_Cost"),
    col("GB_Insurance_Cost_Currency").cast("string").alias("GB_Insurance_Cost_Currency"),
    col("Incoterm_Code").cast("string").alias("Incoterm_Code"),
    col("Conveyance_Reference").cast("string").alias("Conveyance_Reference"),
    col("Additional_Documents_Supporting_Document_Code_1").cast("string").alias("Additional_Documents_Supporting_Document_Code_1"),
    col("Additional_Documents_Supporting_Document_Reference_1").cast("string").alias("Additional_Documents_Supporting_Document_Reference_1"),
    col("Additional_Documents_Complementary_Info_1").cast("string").alias("Additional_Documents_Complementary_Info_1"),
    col("Additional_Documents_Supporting_Document_Code_2").cast("string").alias("Additional_Documents_Supporting_Document_Code_2"),
    col("Additional_Documents_Supporting_Document_Reference_2").cast("string").alias("Additional_Documents_Supporting_Document_Reference_2"),
    col("Additional_Documents_Complementary_Info_2").cast("string").alias("Additional_Documents_Complementary_Info_2"),
    col("Additional_Documents_Supporting_Document_Code_3").cast("string").alias("Additional_Documents_Supporting_Document_Code_3"),
    col("Additional_Documents_Supporting_Document_Reference_3").cast("string").alias("Additional_Documents_Supporting_Document_Reference_3"),
    col("Additional_Documents_Complementary_Info_3").cast("string").alias("Additional_Documents_Complementary_Info_3"),
    col("Additional_Documents_Transit_Document_Code_1").cast("string").alias("Additional_Documents_Transit_Document_Code_1"),
    col("Additional_Documents_Transit_Document_Reference_1").cast("string").alias("Additional_Documents_Transit_Document_Reference_1"),
    col("Additional_Documents_Transit_Document_Code_2").cast("string").alias("Additional_Documents_Transit_Document_Code_2"),
    col("Additional_Documents_Transit_Document_Reference_2").cast("string").alias("Additional_Documents_Transit_Document_Reference_2"),
    col("Additional_Documents_Transit_Document_Code_3").cast("string").alias("Additional_Documents_Transit_Document_Code_3"),
    col("Additional_Documents_Transit_Document_Reference_3").cast("string").alias("Additional_Documents_Transit_Document_Reference_3"),
    col("Additional_Documents_Additional_Reference_Code_1").cast("string").alias("Additional_Documents_Additional_Reference_Code_1"),
    col("Additional_Documents_Additional_Reference_1").cast("string").alias("Additional_Documents_Additional_Reference_1"),
    col("Additional_Documents_Additional_Reference_Code_2").cast("string").alias("Additional_Documents_Additional_Reference_Code_2"),
    col("Additional_Documents_Additional_Reference_2").cast("string").alias("Additional_Documents_Additional_Reference_2"),
    col("Additional_Documents_Additional_Reference_Code_3").cast("string").alias("Additional_Documents_Additional_Reference_Code_3"),
    col("Additional_Documents_Additional_Reference_3").cast("string").alias("Additional_Documents_Additional_Reference_3"),
    col("Location_Name").cast("string").alias("Location_Name"),
    col("Valuation_Indicator").cast("string").alias("Valuation_Indicator"),
    col("Valuation_Method").cast("string").alias("Valuation_Method"),
    col("Estimated_Arrival_Date").cast("string").alias("Estimated_Arrival_Date"),
    col("Estimated_Arrival_Time").cast("string").alias("Estimated_Arrival_Time"),
    col("Additional_Documents_Code_1").cast("string").alias("Additional_Documents_Code_1"),
    col("Additional_Documents_Reference_1").cast("string").alias("Additional_Documents_Reference_1"),
    col("Additional_Documents_Code_2").cast("string").alias("Additional_Documents_Code_2"),
    col("Additional_Documents_Reference_2").cast("string").alias("Additional_Documents_Reference_2"),
    col("Additional_Documents_Code_3").cast("string").alias("Additional_Documents_Code_3"),
    col("Additional_Documents_Reference_3").cast("string").alias("Additional_Documents_Reference_3"),
    col("Additional_Documents_Code_4").cast("string").alias("Additional_Documents_Code_4"),
    col("Additional_Documents_Reference_4").cast("string").alias("Additional_Documents_Reference_4"),
    col("Additional_Documents_Code_5").cast("string").alias("Additional_Documents_Code_5"),
    col("Additional_Documents_Reference_5").cast("string").alias("Additional_Documents_Reference_5"),
    col("Additional_Documents_Code_6").cast("string").alias("Additional_Documents_Code_6"),
    col("Additional_Documents_Reference_6").cast("string").alias("Additional_Documents_Reference_6"),
    col("Commission_and_Brokerage_Value_AB").cast("string").alias("Commission_and_Brokerage_Value_AB"),
    col("Commission_and_Brokerage_Currency_AB").cast("string").alias("Commission_and_Brokerage_Currency_AB"),
    col("Container_and_Packing_Value_AD").cast("string").alias("Container_and_Packing_Value_AD"),
    col("Container_and_Packing_Currency_AD").cast("string").alias("Container_and_Packing_Currency_AD"),
    col("Royalties_and_Licensing_Fees_AI").cast("string").alias("Royalties_and_Licensing_Fees_AI"),
    col("Royalties_and_Licensing_Fees_Currency_AI").cast("string").alias("Royalties_and_Licensing_Fees_Currency_AI"),
    col("Freight_Charges_AK").cast("string").alias("Freight_Charges_AK"),
    col("Freight_Charges_Currency_AK").cast("string").alias("Freight_Charges_Currency_AK"),
    col("Insurance_Costs_AK").cast("string").alias("Insurance_Costs_AK"),
    col("Insurance_Costs_Currency_AK").cast("string").alias("Insurance_Costs_Currency_AK"),
    col("Other_Payments_AL").cast("string").alias("Other_Payments_AL"),
    col("Other_Payments_Currency_AL").cast("string").alias("Other_Payments_Currency_AL"),
    col("Internal_EU_Transport_Value_1X").cast("string").alias("Internal_EU_Transport_Value_1X"),
    col("Internal_EU_Transport_Currency_1X").cast("string").alias("Internal_EU_Transport_Currency_1X"),
    col("Transport_After_Arrival_Value_BA").cast("string").alias("Transport_After_Arrival_Value_BA"),
    col("Transport_After_Arrival_Currency_BA").cast("string").alias("Transport_After_Arrival_Currency_BA"),
    col("Import_Duties_Value_BC").cast("string").alias("Import_Duties_Value_BC"),
    col("Import_Duties_Currency_BC").cast("string").alias("Import_Duties_Currency_BC"),
    col("Buying_Commission_Value_BF").cast("string").alias("Buying_Commission_Value_BF"),
    col("Buying_Commission_Currency_BF").cast("string").alias("Buying_Commission_Currency_BF"),
    substring(col("Movement_Key").cast("string"),1, 25).alias("Movement_Key"),
    col("AIS_Movement").cast("string").alias("AIS_Movement"),
    col("NCTS_Movement").cast("string").alias("NCTS_Movement"),
    col("Declaration_Type").cast("string").alias("Declaration_Type"),
    substring(col("Transaction_Type").cast("string"),1, 3).alias("Transaction_Type"),
    col("Export_Goods_Location").cast("string").alias("Export_Goods_Location"),
    col("NCTS_Location_Type").cast("string").alias("NCTS_Location_Type"),
    col("NCTS_Goods_Location").cast("string").alias("NCTS_Goods_Location"),
    col("AIS_Goods_Location").cast("string").alias("AIS_Goods_Location"),
    col("Customs_Office_of_Departure").cast("string").alias("Customs_Office_of_Departure"),
    col("Customs_Office_of_Transit").cast("string").alias("Customs_Office_of_Transit"),
    col("Customs_Office_of_Transit_2").cast("string").alias("Customs_Office_of_Transit_2"),
    col("Customs_Office_of_Transit_3").cast("string").alias("Customs_Office_of_Transit_3"),
    col("Customs_Office_of_Transit_4").cast("string").alias("Customs_Office_of_Transit_4"),
    col("Customs_Office_of_Transit_5").cast("string").alias("Customs_Office_of_Transit_5"),
    col("Customs_Office_of_Transit_6").cast("string").alias("Customs_Office_of_Transit_6"),
    col("Guarantee_Key").cast("string").alias("Guarantee_Key"),
    col("Freight_Forwarder").cast("string").alias("Freight_Forwarder"),
    col("Consolidator").cast("string").alias("Consolidator"),
    col("Manufacturer").cast("string").alias("Manufacturer"),
    col("Warehousekeeper").cast("string").alias("Warehousekeeper"),
    col("Border_Transport_Type").cast("string").alias("Border_Transport_Type"),
    col("Border_Transport_Mode").cast("string").alias("Border_Transport_Mode"),
    col("Inland_Transport_Mode").cast("string").alias("Inland_Transport_Mode"),
    col("Inland_Transport_Type").cast("string").alias("Inland_Transport_Type"),
    col("Destination_Transport_Mode").cast("string").alias("Destination_Transport_Mode"),
    col("Carrier_Key").cast("string").alias("Carrier_Key"),
    substring(col("Destination_Country").cast("string"),1, 2).alias("Destination_Country"),
    col("Destination_Office").cast("string").alias("Destination_Office"),
    col("Exit_Office").cast("string").alias("Exit_Office"),
    col("Place_of_Loading").cast("string").alias("Place_of_Loading"),
    col("Place_of_Unloading").cast("string").alias("Place_of_Unloading"),
    col("Key").cast("string").alias("Key")
)
if debug:
    display(shipments_1_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 19d89433-ee1d-4611-b062-12108fde3cec)

## Level 2 - Parcel

In [9]:
shipments_2_df = spark.sql(f"""
SELECT 
'2' AS Parcel_Level_No,
whscontainertable.containerid AS Parcel_Reference,
CAST(whscontainertable.weight AS DECIMAL(10,6)) AS Gross_Weight,
CAST(sql_netweight.Netweight AS DECIMAL(10,6)) AS Net_Weight,
'CT' AS Package_Type,
'1' AS Package_Qty,
CASE WHEN salestable.custgroup IN ("Interco", "3rd Party", "RTV") THEN salestable.salesid ELSE whscontainertable.shipcarriertrackingnum END AS Package_Marks,
salestable.salesname AS Consignee_Name,
logisticspostaladdress.street AS Consignee_Street,
logisticspostaladdress.zipcode AS Consignee_Post_Code,
logisticspostaladdress.city AS Consignee_City,
logisticsaddresscountryregion.isocode AS Consignee_Country,
whscontainertable.shipcarriertrackingnum AS Airway_Bill_Reference,
concat_ws("||", whscontainertable.shipmentid, whscontainertable.containerid) AS Key

FROM whscontainertable

INNER JOIN whsshipmenttable
ON whscontainertable.shipmentid = whsshipmenttable.shipmentid
AND whscontainertable.dataareaid = whsshipmenttable.dataareaid
AND whscontainertable.containerstatus = 2

INNER JOIN salestable
ON whsshipmenttable.ordernum = salestable.salesid
AND whsshipmenttable.dataareaid = salestable.dataareaid

LEFT JOIN logisticspostaladdress
ON salestable.deliverypostaladdress = logisticspostaladdress.recid

LEFT JOIN logisticsaddresscountryregion
ON logisticspostaladdress.countryregionid = logisticsaddresscountryregion.countryregionid

LEFT JOIN (select whscontainerline.containerid, sum(whscontainerline.qty * whsloadline.itemnetweight) as Netweight
from whscontainerline  join whsloadline on whscontainerline.loadline = whsloadline.recid group by containerid) as sql_netweight
ON whscontainertable.containerid = sql_netweight.containerid


WHERE whscontainertable.dataareaid IN ('end.', 'END.')
AND logisticspostaladdress.validfrom <= current_date()
AND logisticspostaladdress.validto > current_date()
AND whscontainertable.containerstatus = 2
AND logisticspostaladdress.countryregionid != 'GBR'
AND salestable.dlvmode NOT LIKE '%DPD%'
AND salestable.dlvmode NOT LIKE '%Royal%'
AND logisticspostaladdress.zipcode NOT LIKE 'BF%'

AND whscontainertable.closecontainerutcdatetime >= '{filterdate}'

""")

if debug:
      display(shipments_2_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 10, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 94c51de9-5045-4c91-bdc8-2bd4ac4621fa)

In [10]:
shipments_2_df = shipments_2_df.select(
    substring(col("Parcel_Level_No").cast("string"),1, 6).alias("Parcel_Level_No"),
    substring(col("Parcel_Reference").cast("string"),1, 25).alias("Parcel_Reference"),
    substring(col("Gross_Weight").cast("string"),1, 11).alias("Gross_Weight"),
    substring(col("Net_Weight").cast("string"),1, 11).alias("Net_Weight"),
    substring(col("Package_Type").cast("string"),1, 2).alias("Package_Type"),
    substring(col("Package_Qty").cast("string"),1, 5).alias("Package_Qty"),
    substring(col("Package_Marks").cast("string"),1, 42).alias("Package_Marks"),
    substring(col("Consignee_Name").cast("string"),1, 35).alias("Consignee_Name"),
    substring(col("Consignee_Street").cast("string"),1, 70).alias("Consignee_Street"),
    substring(col("Consignee_Post_Code").cast("string"),1, 9).alias("Consignee_Post_Code"),
    substring(col("Consignee_City").cast("string"),1, 35).alias("Consignee_City"),
    substring(col("Consignee_Country").cast("string"),1, 2).alias("Consignee_Country"),
    substring(col("Airway_Bill_Reference").cast("string"),1, 50).alias("Airway_Bill_Reference"),
    col("Key").cast("string").alias("Key")
)
if debug:
    display(shipments_2_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 11, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 68d6f617-e500-4cb1-a269-b18530d77ebd)

## Level 3 - Parcel Detail

In [11]:
shipments_3_df = spark.sql(f"""
SELECT 
'3' AS Parcel_Detail_Level_No,
whscontainertable.containerid as Line_Reference, -- Remove this later, not required in output
whscontainerline.itemid AS Product_Key,
logisticsaddresscountryregion.isocode AS Country_of_Origin,
CAST(whscontainerline.qty AS DECIMAL(16,4)) AS Item_Qty,
CAST(salesline.costprice AS DECIMAL(12,2)) AS Item_Value,
'GBP' AS Item_Value_Currency,
'' AS Gross_Weight,
'' AS Net_Weight,
'' AS Package_Type,
'' AS Package_Qty,
'' AS Package_Marks,
CASE WHEN ecorescategoryintrastat.additionalunits NOT IN (30,31) OR ecorescategoryintrastat.additionalunits IS NULL THEN 30 ELSE ecorescategoryintrastat.additionalunits END AS Qty_Code,
CAST((whscontainerline.qty / salesline.salesqty) * salesline.lineamount AS DECIMAL(12,2)) AS Destination_Value,
salesline.currencycode AS Destination_Currency,
'' AS Export_Value,
'' AS Export_Value_Currency,
'' AS Additional_Documents_Supporting_Document_Code_1,
'' AS Additional_Documents_Supporting_Document_Reference_1,
'' AS Additional_Documents_Complementary_Info_1,
'' AS Additional_Documents_Supporting_Document_Code_2,
'' AS Additional_Documents_Supporting_Document_Reference_2,
'' AS Additional_Documents_Complementary_Info_2,
'' AS Additional_Documents_Supporting_Document_Code_3,
'' AS Additional_Documents_Supporting_Document_Reference_3,
'' AS Additional_Documents_Complementary_Info_3,
'' AS Additional_Documents_Transit_Document_Code_1,
'' AS Additional_Documents_Transit_Document_Reference_1,
'' AS Additional_Documents_Transit_Document_Code_2,
'' AS Additional_Documents_Transit_Document_Reference_2,
'' AS Additional_Documents_Transit_Document_Code_3,
'' AS Additional_Documents_Transit_Document_Reference_3,
'' AS Additional_Documents_Additional_Reference_Code_1,
'' AS Additional_Documents_Additional_Reference_1,
'' AS Additional_Documents_Additional_Reference_Code_2,
'' AS Additional_Documents_Additional_Reference_2,
'' AS Additional_Documents_Additional_Reference_Code_3,
'' AS Additional_Documents_Additional_Reference_3,
'' AS Preference_Country,
'' AS Preference_Document_Code,
'' AS Preference_Document_Reference,
'' AS Additional_Documents_Code_1,
'' AS Additional_Documents_Reference_1,
'' AS Additional_Documents_Code_2,
'' AS Additional_Documents_Reference_2,
'' AS Additional_Documents_Code_3,
'' AS Additional_Documents_Reference_3,
'' AS Additional_Documents_Code_4,
'' AS Additional_Documents_Reference_4,
'' AS Additional_Documents_Code_5,
'' AS Additional_Documents_Reference_5,
'' AS Additional_Documents_Code_6,
'' AS Additional_Documents_Reference_6,
concat_ws("||", whscontainerline.shipmentid, whscontainerline.containerid, whscontainerline.recid) AS Key

FROM whscontainerline

INNER JOIN whsshipmenttable
ON whscontainerline.shipmentid = whsshipmenttable.shipmentid
AND whscontainerline.dataareaid = whsshipmenttable.dataareaid

INNER JOIN whscontainertable
ON whscontainerline.containerid = whscontainertable.containerid
AND whscontainerline.dataareaid = whscontainertable.dataareaid
AND whscontainertable.containerstatus = 2

INNER JOIN salestable
ON whsshipmenttable.ordernum = salestable.salesid
AND whsshipmenttable.dataareaid = salestable.dataareaid

INNER JOIN whsloadline
ON whscontainerline.loadline = whsloadline.recid
AND whscontainerline.dataareaid = whsloadline.dataareaid

INNER JOIN salesline
ON whsloadline.inventtransid = salesline.inventtransid
AND whsloadline.dataareaid = salesline.dataareaid

LEFT JOIN inventtable
ON whsloadline.itemid = inventtable.itemid
AND whsloadline.dataareaid = inventtable.dataareaid

LEFT JOIN ecorescategoryintrastat
ON inventtable.intrastatcommodity = ecorescategoryintrastat.category

LEFT JOIN logisticsaddresscountryregion
ON inventtable.origcountryregionid = logisticsaddresscountryregion.countryregionid

LEFT JOIN logisticspostaladdress
ON salestable.deliverypostaladdress = logisticspostaladdress.recid

WHERE whscontainerline.dataareaid IN ('end.', 'END.')
AND whscontainertable.containerstatus = 2
AND logisticspostaladdress.validfrom <= current_date()
AND logisticspostaladdress.validto > current_date()
AND logisticspostaladdress.countryregionid != 'GBR'
AND salestable.dlvmode NOT LIKE '%DPD%'
AND salestable.dlvmode NOT LIKE '%Royal%'
AND logisticspostaladdress.zipcode NOT LIKE 'BF%'

AND whscontainertable.closecontainerutcdatetime >= '{filterdate}'
""")

if debug:
      display(shipments_3_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 12, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 737b87f5-ee2c-4e16-ae48-717ff59d027b)

In [12]:
shipments_3_df = shipments_3_df.select(
    substring(col("Parcel_Detail_Level_No").cast("string"),1, 12).alias("Parcel_Detail_Level_No"),
    col("Line_Reference").cast("string").alias("Line_Reference"),
    substring(col("Product_Key").cast("string"),1, 25).alias("Product_Key"),
    substring(col("Country_of_Origin").cast("string"),1, 2).alias("Country_of_Origin"),
    substring(col("Item_Qty").cast("string"),1, 17).alias("Item_Qty"),
    substring(col("Item_Value").cast("string"),1, 13).alias("Item_Value"),
    substring(col("Item_Value_Currency").cast("string"),1, 3).alias("Item_Value_Currency"),
    col("Gross_Weight").cast("string").alias("Gross_Weight"),
    col("Net_Weight").cast("string").alias("Net_Weight"),
    col("Package_Type").cast("string").alias("Package_Type"),
    col("Package_Qty").cast("string").alias("Package_Qty"),
    col("Package_Marks").cast("string").alias("Package_Marks"),
    col("Qty_Code").cast("string").alias("Qty_Code"),
    substring(col("Destination_Value").cast("string"),1, 13).alias("Destination_Value"),
    substring(col("Destination_Currency").cast("string"),1, 3).alias("Destination_Currency"),
    col("Export_Value").cast("string").alias("Export_Value"),
    col("Export_Value_Currency").cast("string").alias("Export_Value_Currency"),
    col("Additional_Documents_Supporting_Document_Code_1").cast("string").alias("Additional_Documents_Supporting_Document_Code_1"),
    col("Additional_Documents_Supporting_Document_Reference_1").cast("string").alias("Additional_Documents_Supporting_Document_Reference_1"),
    col("Additional_Documents_Complementary_Info_1").cast("string").alias("Additional_Documents_Complementary_Info_1"),
    col("Additional_Documents_Supporting_Document_Code_2").cast("string").alias("Additional_Documents_Supporting_Document_Code_2"),
    col("Additional_Documents_Supporting_Document_Reference_2").cast("string").alias("Additional_Documents_Supporting_Document_Reference_2"),
    col("Additional_Documents_Complementary_Info_2").cast("string").alias("Additional_Documents_Complementary_Info_2"),
    col("Additional_Documents_Supporting_Document_Code_3").cast("string").alias("Additional_Documents_Supporting_Document_Code_3"),
    col("Additional_Documents_Supporting_Document_Reference_3").cast("string").alias("Additional_Documents_Supporting_Document_Reference_3"),
    col("Additional_Documents_Complementary_Info_3").cast("string").alias("Additional_Documents_Complementary_Info_3"),
    col("Additional_Documents_Transit_Document_Code_1").cast("string").alias("Additional_Documents_Transit_Document_Code_1"),
    col("Additional_Documents_Transit_Document_Reference_1").cast("string").alias("Additional_Documents_Transit_Document_Reference_1"),
    col("Additional_Documents_Transit_Document_Code_2").cast("string").alias("Additional_Documents_Transit_Document_Code_2"),
    col("Additional_Documents_Transit_Document_Reference_2").cast("string").alias("Additional_Documents_Transit_Document_Reference_2"),
    col("Additional_Documents_Transit_Document_Code_3").cast("string").alias("Additional_Documents_Transit_Document_Code_3"),
    col("Additional_Documents_Transit_Document_Reference_3").cast("string").alias("Additional_Documents_Transit_Document_Reference_3"),
    col("Additional_Documents_Additional_Reference_Code_1").cast("string").alias("Additional_Documents_Additional_Reference_Code_1"),
    col("Additional_Documents_Additional_Reference_1").cast("string").alias("Additional_Documents_Additional_Reference_1"),
    col("Additional_Documents_Additional_Reference_Code_2").cast("string").alias("Additional_Documents_Additional_Reference_Code_2"),
    col("Additional_Documents_Additional_Reference_2").cast("string").alias("Additional_Documents_Additional_Reference_2"),
    col("Additional_Documents_Additional_Reference_Code_3").cast("string").alias("Additional_Documents_Additional_Reference_Code_3"),
    col("Additional_Documents_Additional_Reference_3").cast("string").alias("Additional_Documents_Additional_Reference_3"),
    col("Preference_Country").cast("string").alias("Preference_Country"),
    col("Preference_Document_Code").cast("string").alias("Preference_Document_Code"),
    col("Preference_Document_Reference").cast("string").alias("Preference_Document_Reference"),
    col("Additional_Documents_Code_1").cast("string").alias("Additional_Documents_Code_1"),
    col("Additional_Documents_Reference_1").cast("string").alias("Additional_Documents_Reference_1"),
    col("Additional_Documents_Code_2").cast("string").alias("Additional_Documents_Code_2"),
    col("Additional_Documents_Reference_2").cast("string").alias("Additional_Documents_Reference_2"),
    col("Additional_Documents_Code_3").cast("string").alias("Additional_Documents_Code_3"),
    col("Additional_Documents_Reference_3").cast("string").alias("Additional_Documents_Reference_3"),
    col("Additional_Documents_Code_4").cast("string").alias("Additional_Documents_Code_4"),
    col("Additional_Documents_Reference_4").cast("string").alias("Additional_Documents_Reference_4"),
    col("Additional_Documents_Code_5").cast("string").alias("Additional_Documents_Code_5"),
    col("Additional_Documents_Reference_5").cast("string").alias("Additional_Documents_Reference_5"),
    col("Additional_Documents_Code_6").cast("string").alias("Additional_Documents_Code_6"),
    col("Additional_Documents_Reference_6").cast("string").alias("Additional_Documents_Reference_6"),
    col("Key").cast("string").alias("Key")
)
if debug:
    display(shipments_3_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 13, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 6f52f5fe-6e7f-4850-8471-386f16208478)

## Date Field Changing Post Query(s)

In [13]:
list_date_columns_1 = [name for name, dtype in shipments_1_df.dtypes if dtype in ('date','timestamp')]
list_date_columns_2 = [name for name, dtype in shipments_2_df.dtypes if dtype in ('date','timestamp')]
list_date_columns_3 = [name for name, dtype in shipments_3_df.dtypes if dtype in ('date','timestamp')]

if debug:
    print("Date Columns to change: " , list_date_columns_1)
    print("Date Columns to change: " , list_date_columns_2)
    print("Date Columns to change: " , list_date_columns_3)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 14, Finished, Available, Finished)

Date Columns to change:  ['Date_of_Shipment']
Date Columns to change:  []
Date Columns to change:  []


In [14]:
for column in list_date_columns_1:
    shipments_1_df = shipments_1_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

for column in list_date_columns_2:
    shipments_2_df = shipments_2_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

for column in list_date_columns_3:
    shipments_3_df = shipments_3_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 15, Finished, Available, Finished)

# Regex Pass to Remove Non-ASCII

In [15]:
shipments_2_regex_columns = ["Consignee_Name", "Consignee_Street", "Consignee_Post_Code", "Consignee_City"]

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 16, Finished, Available, Finished)

In [16]:
for column in shipments_2_regex_columns:
    shipments_2_df = shipments_2_df.withColumn(column, trim(regexp_replace(regexp_replace(col(column), "[^\\x00-\\x7F]|[\\|:#/?,Â%(),.]", ""), "\\s+", " ")))

if debug:
    display(shipments_2_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 17, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, c598543b-57d9-4824-85c7-36ebc5831529)

## Display Pre-Filtering

In [17]:
if debug:
    display(shipments_1_df)
    display(shipments_2_df)
    display(shipments_3_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 18, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, c7aedaf5-801b-44a2-83e2-2c15ab5f39d2)

SynapseWidget(Synapse.DataFrame, 3f4077ff-4ada-4234-a16a-4fa3f2ede1d5)

SynapseWidget(Synapse.DataFrame, 13f05b73-4baa-41e9-8024-45ef04c3b015)

# Init Error Check Process Level 1

In [18]:
shipments_1_Mandatory_Columns = [
    "Shipment_Level_No",
    "Shipment_Reference",
    "Date_of_Shipment",
    "Project_Key",
    "NOTC_A",
    "Border_Transport_ID",
    "Border_Transport_Nationality",
    "Inland_Transport_ID",
    "Incoterm_Code",
    "Location_Name",
    "Valuation_Method",
    "Estimated_Arrival_Date",
    "Estimated_Arrival_Time",
    "Movement_Key",
    "Declaration_Type",
    "Transaction_Type",
    "Export_Goods_Location",
    "NCTS_Location_Type",
    "NCTS_Goods_Location",
    "AIS_Goods_Location",
    "Customs_Office_of_Departure",
    "Guarantee_Key",
    "Border_Transport_Mode",
    "Inland_Transport_Mode",
    "Destination_Country",
    "Destination_Office",
    "Exit_Office",
    "Place_of_Loading"
]

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 19, Finished, Available, Finished)

In [19]:
level = 1
null_condition = None
null_column_names_exprs = []

# Build null checks and collect column names
for column in shipments_1_Mandatory_Columns:
    condition = col(column).isNull()
    null_condition = condition if null_condition is None else null_condition | condition
    null_column_names_exprs.append(when(condition, lit(column)))

# Create column for failed columns
shipments_1_with_errors = shipments_1_df.withColumn(
    "failed_columns",
    array(*null_column_names_exprs)
)

# Build error message string
shipments_1_with_errors = shipments_1_with_errors.withColumn(
    "error_fields",
    when(size(col("failed_columns")) > 0,
         concat_ws("", lit("Columns "), concat_ws(", ", col("failed_columns")), lit(f" are null at level {level}")))
)

# Split into good and bad
shipments_1_bad_df = shipments_1_with_errors.filter(null_condition).drop("failed_columns")
shipments_1_df = shipments_1_with_errors.filter(~null_condition).drop("error_fields", "failed_columns")

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 20, Finished, Available, Finished)

In [20]:
if debug:
    display(shipments_1_bad_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 21, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 5f69adcc-2d94-40a7-984b-5f9824213261)

# Init Error Check Process Level 2

In [21]:
shipments_2_Mandatory_Columns = [
    "Parcel_Level_No",
    "Parcel_Reference",
    "Gross_Weight",
    "Net_Weight",
    "Package_Type",
    "Package_Qty"
]

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 22, Finished, Available, Finished)

In [22]:
level = 2
null_condition = None
null_column_names_exprs = []

for column in shipments_2_Mandatory_Columns:
    condition = col(column).isNull()
    null_condition = condition if null_condition is None else null_condition | condition
    null_column_names_exprs.append(when(condition, lit(column)))

# Join level 1 Shipment_Reference with level 2 Parcel_Reference
shipments_2_with_join = shipments_2_df.join(
    shipments_1_bad_df.selectExpr("Shipment_Reference", "error_fields as level1_errors"),
    shipments_2_df["Parcel_Reference"] == col("Shipment_Reference"),
    how="left"
)

# Build array of failed columns
shipments_2_with_errors = shipments_2_with_join.withColumn(
    "level2_failed_columns", array(*null_column_names_exprs)
)

# Remove nulls from the array
shipments_2_with_errors = shipments_2_with_errors.withColumn(
    "level2_nonnull_failed_columns",
    expr("filter(level2_failed_columns, x -> x is not null)")
)

# Create level 2 error message
shipments_2_with_errors = shipments_2_with_errors.withColumn(
    "level2_error_fields",
    when(
        size(col("level2_nonnull_failed_columns")) > 0,
        concat_ws("", lit("Columns "), concat_ws(", ", col("level2_nonnull_failed_columns")), lit(f" are null at level {level}"))
    )
)

# Combine level 1 and level 2 error messages
shipments_2_with_errors = shipments_2_with_errors.withColumn(
    "error_fields",
    concat_ws("; ",
        *[col(c) for c in ["level1_errors", "level2_error_fields"] if c in shipments_2_with_errors.columns]
    )
)

# Split into bad and good rows
shipments_2_bad_df = shipments_2_with_errors.filter(null_condition | col("level1_errors").isNotNull()) \
    .drop("level1_errors", "level2_failed_columns", "level2_nonnull_failed_columns", "level2_error_fields","Shipment_Reference")

shipments_2_df = shipments_2_with_errors.filter(~(null_condition | col("level1_errors").isNotNull())) \
    .drop("level1_errors", "level2_failed_columns", "level2_nonnull_failed_columns", "level2_error_fields", "error_fields","Shipment_Reference")

# Go back and remove any bad Shipment_References from level 1 good that are bad in level 2
# Perform a left anti join to exclude the bad Parcel_References from level 2
shipments_1_df = shipments_1_df.join(
    shipments_2_bad_df.select("Parcel_Reference"),
    shipments_1_df["Shipment_Reference"] == shipments_2_bad_df["Parcel_Reference"],
    how="left_anti"
)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 23, Finished, Available, Finished)

In [23]:
if debug:
    display(shipments_2_bad_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 24, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, b931175e-17a1-4d89-b5f0-42a791014f9a)

# Init Error Check Process Level 3

In [24]:
shipments_3_Mandatory_Columns = [
    "Parcel_Detail_Level_No",
    "Product_Key",
    "Country_of_Origin",
    "Item_Qty",
    "Item_Value",
    "Item_Value_Currency",
    "Gross_Weight",
    "Net_Weight",
    "Package_Type",
    "Package_Qty",
    "Package_Marks",
    "Qty_Code"
]

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 25, Finished, Available, Finished)

In [25]:
level = 3
null_condition = None
null_column_names_exprs = []

# Build null checks and collect column names for level 3
for column in shipments_3_Mandatory_Columns:
    condition = F.col(column).isNull()
    null_condition = condition if null_condition is None else null_condition | condition
    null_column_names_exprs.append(F.when(condition, F.lit(column)))

# Join level 1 Shipment_Reference with level 3 Line_Reference
shipments_3_with_join = shipments_3_df.join(
    shipments_1_bad_df.selectExpr("Shipment_Reference as Shipment_Reference_1", "error_fields as level1_errors"),
    shipments_3_df["Line_Reference"] == F.col("Shipment_Reference_1"),
    how="left"
).join(
    shipments_2_bad_df.selectExpr("Parcel_Reference as Parcel_Reference_2", "error_fields as level2_errors"),
    shipments_3_df["Line_Reference"] == F.col("Parcel_Reference_2"),
    how="left"
)

# Build array of failed columns for level 3
shipments_3_with_errors = shipments_3_with_join.withColumn(
    "level3_failed_columns", F.array(*null_column_names_exprs)
)

# Remove nulls from the failed columns array
shipments_3_with_errors = shipments_3_with_errors.withColumn(
    "level3_nonnull_failed_columns",
    F.expr("filter(level3_failed_columns, x -> x is not null)")
)

# Create level 3 error message
shipments_3_with_errors = shipments_3_with_errors.withColumn(
    "level3_error_fields",
    F.when(
        F.size(F.col("level3_nonnull_failed_columns")) > 0,
        F.concat_ws("", F.lit("Columns "), F.concat_ws(", ", F.col("level3_nonnull_failed_columns")), F.lit(f" are null at level {level}"))
    ).otherwise(F.lit(""))
)

# Combine error messages from level 1, level 2, and level 3
# First, create a list of error messages from the levels
level_columns = []
for c in ["level1_errors", "level2_errors", "level3_error_fields"]:
    if c in shipments_3_with_errors.columns:
        level_columns.append(F.col(c))

# Concatenate the error messages and remove duplicates
shipments_3_with_errors = shipments_3_with_errors.withColumn(
    "error_fields",
    F.concat_ws("; ", *level_columns)
)

# Remove duplicates by filtering out repeated error messages
# We can use a `distinct` operation in the `concat_ws` to ensure no duplicates
shipments_3_with_errors = shipments_3_with_errors.withColumn(
    "error_fields",
    F.expr("concat_ws('; ', array_distinct(split(error_fields, '; ')))")
)

# Split into bad and good rows
# Use & (bitwise and) instead of logical `and` to combine conditions
shipments_3_bad_df = shipments_3_with_errors.filter(
    (null_condition | F.col("level1_errors").isNotNull() | F.col("level2_errors").isNotNull())
).drop(
    "level1_errors", "level2_errors", "level3_failed_columns", "level3_nonnull_failed_columns", "level3_error_fields","Shipment_Reference_1","Parcel_Reference_2"
)

shipments_3_df = shipments_3_with_errors.filter(
    ~(null_condition | F.col("level1_errors").isNotNull() | F.col("level2_errors").isNotNull())
).drop(
    "level1_errors", "level2_errors", "level3_failed_columns", "level3_nonnull_failed_columns", "level3_error_fields", "error_fields","Shipment_Reference_1","Parcel_Reference_2"
)

# Go back and remove any bad Shipment_References or Parcel_References from level 1 and level 2 good based on bad Line_Reference in level 3

# Perform a left anti join to exclude the bad Line_References in level 3
shipments_1_df = shipments_1_df.join(
    shipments_3_bad_df.select("Line_Reference"),  # Select the bad Line_Reference from level 3
    shipments_1_df["Shipment_Reference"] == shipments_3_bad_df["Line_Reference"],  # Match with Shipment_Reference
    how="left_anti"  # Perform a left anti join to exclude the bad Shipment_References
)

shipments_2_df = shipments_2_df.join(
    shipments_3_bad_df.select("Line_Reference"),  # Select the bad Line_Reference from level 3
    shipments_2_df["Parcel_Reference"] == shipments_3_bad_df["Line_Reference"],  # Match with Parcel_Reference
    how="left_anti"  # Perform a left anti join to exclude the bad Parcel_References
)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 26, Finished, Available, Finished)

In [26]:
if debug:
    display(shipments_3_bad_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 27, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 1534a308-ada9-4566-a720-6fc60bdd742d)

In [27]:
shipments_1_bad_keys = [row["Shipment_Reference"] for row in shipments_1_bad_df.select("Shipment_Reference").distinct().collect()]
shipments_2_bad_keys = [row["Parcel_Reference"] for row in shipments_2_bad_df.select("Parcel_Reference").distinct().collect()]
shipments_3_bad_keys = [row["Line_Reference"] for row in shipments_3_bad_df.select("Line_Reference").distinct().collect()]

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 28, Finished, Available, Finished)

In [28]:
if debug:
    print("Level 1 Bad: " , shipments_1_bad_keys)
    print("Level 2 Bad: " , shipments_2_bad_keys)
    print("Level 3 Bad: " , shipments_3_bad_keys)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 29, Finished, Available, Finished)

Level 1 Bad:  []
Level 2 Bad:  []
Level 3 Bad:  []


# Before Relational Bad Row Removal

In [29]:
if debug:
    display(shipments_1_bad_df)
    display(shipments_2_bad_df)
    display(shipments_3_bad_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 30, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, fc2d9ca2-c87b-4b11-8335-3e2a870c372c)

SynapseWidget(Synapse.DataFrame, 8ad0735a-54f9-4464-a404-3c303c8180cf)

SynapseWidget(Synapse.DataFrame, 5b2acaa2-da8c-4b38-a7e4-c5211a1233b9)

In [30]:
shipments_2_df = shipments_2_df.filter(
    ~col("Parcel_Reference").isin(shipments_1_bad_keys)
)

shipments_2_df = shipments_2_df.filter(
    ~col("Parcel_Reference").isin(shipments_3_bad_keys)
)


shipments_3_df = shipments_3_df.filter(
    ~col("Line_Reference").isin(shipments_1_bad_keys)
)

shipments_3_df = shipments_3_df.filter(
    ~col("Line_Reference").isin(shipments_2_bad_keys)
)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 31, Finished, Available, Finished)

## After Relational Bad Row Removal

In [31]:
if debug:
    display(shipments_1_bad_df)
    display(shipments_2_bad_df)
    display(shipments_3_bad_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 32, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, f70d9ae5-ecfa-4031-90b5-646fd9ec8f73)

SynapseWidget(Synapse.DataFrame, 53b74ca1-470a-4424-9367-677f9e1fd986)

SynapseWidget(Synapse.DataFrame, 8cccf020-4c2c-4a80-b594-384a8ef0cf75)

## If it's in Level_1 bad, then is it in in Level_2 & Level_3 good (it should NOT be)

In [32]:
if debug:
    display(shipments_1_bad_df)
    display(shipments_2_df)
    display(shipments_3_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 33, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, bc74c8f7-f931-45cb-91fe-027d601d6c1b)

SynapseWidget(Synapse.DataFrame, e6b7b05c-6fb8-4757-8ab6-2eae01dec674)

SynapseWidget(Synapse.DataFrame, df212ce3-0296-4fce-b9d3-344c3f333a9a)

## If it's in Level_2 bad, then is it in in Level_1 & Level_3 good (it should NOT be)

In [33]:
if debug:
    display(shipments_2_bad_df)
    display(shipments_1_df)
    display(shipments_3_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 34, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 05f95de0-2140-4287-9031-b7a45400e803)

SynapseWidget(Synapse.DataFrame, ffc35052-6028-495d-9226-52c47529d2c0)

SynapseWidget(Synapse.DataFrame, 2f8fb6d9-d412-4f24-a252-a07ea5ce9e07)

## If it's in Level_3 bad, then is it in in Level_1 & Level_2 good (it should NOT be)

In [34]:
if debug:
    display(shipments_3_bad_df)
    display(shipments_1_df)
    display(shipments_2_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 35, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, dc2f379f-c164-4ee0-b40d-49158ef3322b)

SynapseWidget(Synapse.DataFrame, 15c13af4-1a79-4587-8f3e-5373d224f865)

SynapseWidget(Synapse.DataFrame, 63d64238-0dd1-4f7b-895e-967dd343ba98)

# Init Good File Name & Date Logic

In [35]:
file_path_folder = "/lakehouse/default/Files/Output/"
file_extention = '.dat'

# file date time - stamp tomorrow's date if after 6.15pm --Nick: had to knock it back 1 hour to account for timezone difference; working
now = datetime.now()
cutoff_time = now.replace(hour=17, minute=15, second=0, microsecond=0)

if now > cutoff_time:
    tomorrow = now + timedelta(days=1)
    #file_datetime = tomorrow.strftime('%Y-%m-%d')
    file_datetime = tomorrow.strftime('%Y-%m-%d-%H')
else:
    #file_datetime = now.strftime('%Y-%m-%d')
    file_datetime = now.strftime('%Y-%m-%d-%H')


file_name = "shipments" + "_" + file_datetime + file_extention
file_path = file_path_folder + file_name

file_name_temp = "shipments" + "_temp_" + file_datetime + file_extention
file_path_temp = file_path_folder + file_name_temp

file_name_1 = "shipments_1" + "_" + file_datetime + file_extention
file_path_1 = file_path_folder + file_name_1

file_name_2 = "shipments_2" + "_" + file_datetime + file_extention
file_path_2 = file_path_folder + file_name_2

file_name_3 = "shipments_3" + "_" + file_datetime + file_extention
file_path_3 = file_path_folder + file_name_3

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 36, Finished, Available, Finished)

In [36]:
if debug:
    print("Now: " , now)
    print("Cutoff: " , cutoff_time)
    print("File Date: " , file_datetime)
    print("File Folder Path: " , file_path_folder)
    print("Good File Name: " , file_name)
    print("Good File Name: " , file_path)
    print("Good File Name_1: " , file_name_1)
    print("Good File Name_1: " , file_path_1)
    print("Good File Name_2: " , file_name_2)
    print("Good File Name_2: " , file_path_2)
    print("Good File Name_3: " , file_name_3)
    print("Good File Name_3: " , file_path_3)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 37, Finished, Available, Finished)

Now:  2025-05-09 12:07:08.662687
Cutoff:  2025-05-09 17:15:00
File Date:  2025-05-09-12
File Folder Path:  /lakehouse/default/Files/Output/
Good File Name:  shipments_2025-05-09-12.dat
Good File Name:  /lakehouse/default/Files/Output/shipments_2025-05-09-12.dat
Good File Name_1:  shipments_1_2025-05-09-12.dat
Good File Name_1:  /lakehouse/default/Files/Output/shipments_1_2025-05-09-12.dat
Good File Name_2:  shipments_2_2025-05-09-12.dat
Good File Name_2:  /lakehouse/default/Files/Output/shipments_2_2025-05-09-12.dat
Good File Name_3:  shipments_3_2025-05-09-12.dat
Good File Name_3:  /lakehouse/default/Files/Output/shipments_3_2025-05-09-12.dat


# Init Good Row Merge

In [37]:
file_name = "shipments" + "_" + file_datetime + file_extention
file_path = file_path_folder + file_name

if debug:
    print("file: ", file_name, " path: " , file_path)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 38, Finished, Available, Finished)

file:  shipments_2025-05-09-12.dat  path:  /lakehouse/default/Files/Output/shipments_2025-05-09-12.dat


In [38]:
if incremental_run:
    
    # THIS IS ONLY FOR RECORD TRACKING BEFORE KEY COLUMNS ARE DROPPED

    def flatten_and_combine(df, container_column):
        """Concats all column values into a single string, adds a timestamp, Key column, and Container column."""
        key_column = df.columns[-1]  # Assuming 'Key' is the last column

        return df.withColumn("Timestamp", F.current_timestamp()) \
                .withColumn("CombinedText", F.concat_ws("|", *df.columns[:-1])) \
                .withColumn("Container", F.col(container_column)) \
                .select("Timestamp", key_column, "Container", "CombinedText")

    # Apply to each DataFrame with the appropriate container column
    df1_flat = flatten_and_combine(shipments_1_df, "Shipment_Reference")
    df2_flat = flatten_and_combine(shipments_2_df, "Parcel_Reference")
    df3_flat = flatten_and_combine(shipments_3_df, "Line_Reference")

    # Combine them all
    combined_df = df1_flat.unionByName(df2_flat).unionByName(df3_flat)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 39, Finished, Available, Finished)

In [39]:
if debug & incremental_run:
    display(combined_df)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 40, Finished, Available, Finished)

In [40]:
if incremental_run:
    
    # REMOVE ROWS FROM CURRENT RUN THAT HAVE ALREADY BEEN SENT (EXIST IN RECORD TRACKING)

    lakehouse_table_name = "bondedwarehouserecordtracking_shipments"
    container_column = "Container"

    try:
        # Loads already sent Containers from record tracking
        sentrecords_df = spark.read.table(lakehouse_table_name).select(container_column).distinct()

        # Filter each input dataframe to EXCLUDE already sent
        shipments_1_df = shipments_1_df.join(sentrecords_df, shipments_1_df["Shipment_Reference"] == sentrecords_df["Container"], "left_anti")
        shipments_2_df = shipments_2_df.join(sentrecords_df, shipments_2_df["Parcel_Reference"] == sentrecords_df["Container"], "left_anti")
        shipments_3_df = shipments_3_df.join(sentrecords_df, shipments_3_df["Line_Reference"] == sentrecords_df["Container"], "left_anti")

        # flatten and combine to add current rows to record tracking table later
        df1_flat = flatten_and_combine(shipments_1_df, "Shipment_Reference")
        df2_flat = flatten_and_combine(shipments_2_df, "Parcel_Reference")
        df3_flat = flatten_and_combine(shipments_3_df, "Line_Reference")

        combined_df = df1_flat.unionByName(df2_flat).unionByName(df3_flat)

    except Exception as e:
        print(f"An error occurred: {e}")

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 41, Finished, Available, Finished)

# Export Good

In [41]:
save_dataframe_to_csv(shipments_1_df.drop("Key","Level_Key","Parent_Key"), file_path_1, show_header=False)
save_dataframe_to_csv(shipments_2_df.drop("Key","Level_Key","Parent_Key"), file_path_2, show_header=False)
save_dataframe_to_csv(shipments_3_df.drop("Key","Level_Key","Parent_Key"), file_path_3, show_header=False)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 42, Finished, Available, Finished)

Add pipes: True
Show headers: False
Add pipes: True
Show headers: False
Add pipes: True
Show headers: False


In [42]:
dat_file_paths = [
    file_path_1,
    file_path_2,
    file_path_3
]

if debug:
    list(dat_file_paths)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 43, Finished, Available, Finished)

In [43]:
try:
    shipments_1_dat = pd.read_csv(file_path_1, sep='|', header=None, dtype=str)
    shipments_2_dat = pd.read_csv(file_path_2, sep='|', header=None, dtype=str)
    shipments_3_dat = pd.read_csv(file_path_3, sep='|', header=None, dtype=str)
    good_rows = True

except:
    print("Empty file here at GOOD shipments level")
    good_rows = False
    ready_to_copy = False

    try:
        for file in dat_file_paths:
            os.remove(file)
            print(f"Deleted: {file}")
    except:
        print("Tried to delete part files, and failed.")

if debug:
    display(shipments_1_dat)
    display(shipments_2_dat)
    display(shipments_3_dat)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 44, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, f731817c-e5fb-4413-a7aa-156bf3f940e1)

SynapseWidget(Synapse.DataFrame, 7e5d6366-af70-4d29-b634-ea8cc91dfe9c)

SynapseWidget(Synapse.DataFrame, a9c78d8b-a3fb-4f80-9916-a8b1d864b0a8)

In [44]:
if good_rows:
    
    shipments_123_dat = pd.concat([shipments_1_dat, shipments_2_dat, shipments_3_dat], ignore_index=True)
    shipments_123_dat = shipments_123_dat.sort_values(by=[1, 0]).reset_index(drop=True)

    shipments_123_dat[0] = shipments_123_dat[0].replace({
        '1': 'SHIPMENT',
        '2': 'PARCEL',
        '3': 'PARCELDETAIL'
    })

    # Convert the DataFrame back to CSV format
    csv_data_before = shipments_123_dat.to_csv(sep='|', index=False, header=False)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 45, Finished, Available, Finished)

In [45]:
if good_rows:
    
    with open(file_path, 'w') as file:
        file.write(csv_data_before)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 46, Finished, Available, Finished)

In [46]:
for file in dat_file_paths:
    os.remove(file)
    print(f"Deleted: {file}")

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 47, Finished, Available, Finished)

Deleted: /lakehouse/default/Files/Output/shipments_1_2025-05-09-12.dat
Deleted: /lakehouse/default/Files/Output/shipments_2_2025-05-09-12.dat
Deleted: /lakehouse/default/Files/Output/shipments_3_2025-05-09-12.dat


In [47]:
if good_rows:
        
    input_path = file_path
    temp_output_path = file_path_temp

    with open(input_path, "r", encoding="utf-8") as infile, \
         open(temp_output_path, "w", encoding="utf-8") as outfile:
        
        for line in infile:
            line = line.strip()

            if line.startswith("SHIPMENT|"):
                line = line.replace("N A", "N/A")
                
            elif line.startswith("PARCEL|"):
                line = re.sub(r'\|+$', '', line) + '|'

            elif line.startswith("PARCELDETAIL|"):
                parts = line.split("|")
                if len(parts) > 2:
                    del parts[1]
                line = "|".join(parts)
                line = re.sub(r'\|+$', '', line) + '|' * 39

            outfile.write(line + '\n')

    os.replace(temp_output_path, input_path)

    ready_to_copy = True

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 48, Finished, Available, Finished)

# Init Bad File Name

In [48]:
error_file_1 = "shipments_1_errors_" + file_datetime + file_extention
error_file_path_1 = file_path_folder + error_file_1

error_file_2 = "shipments_2_errors_" + file_datetime + file_extention
error_file_path_2 = file_path_folder + error_file_2

error_file_3 = "shipments_3_errors_" + file_datetime + file_extention
error_file_path_3 = file_path_folder + error_file_3

if debug:
    print("file: ", error_file_1, " path: " , error_file_path_1)
    print("file: ", error_file_2, " path: " , error_file_path_2)
    print("file: ", error_file_3, " path: " , error_file_path_3)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 49, Finished, Available, Finished)

file:  shipments_1_errors_2025-05-09-12.dat  path:  /lakehouse/default/Files/Output/shipments_1_errors_2025-05-09-12.dat
file:  shipments_2_errors_2025-05-09-12.dat  path:  /lakehouse/default/Files/Output/shipments_2_errors_2025-05-09-12.dat
file:  shipments_3_errors_2025-05-09-12.dat  path:  /lakehouse/default/Files/Output/shipments_3_errors_2025-05-09-12.dat


# Export Bad

In [49]:
save_dataframe_to_csv(shipments_1_bad_df.drop("Key","Level_Key","Parent_Key"), error_file_path_1, show_header = True)
save_dataframe_to_csv(shipments_2_bad_df.drop("Key","Level_Key","Parent_Key"), error_file_path_2, show_header = True)
save_dataframe_to_csv(shipments_3_bad_df.drop("Key","Level_Key","Parent_Key"), error_file_path_3, show_header = True)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 50, Finished, Available, Finished)

Add pipes: True
Show headers: True
Add pipes: True
Show headers: True
Add pipes: True
Show headers: True


# Init Record Tracking

In [50]:
if incremental_run == True and good_rows == True:

    # Save the final_df to different tables based on the exportfile variable value

    def record_tracking_df_to_table(dataframe, table_name, file_name):
        """Saves distinct records to a table, checking for duplicates and enabling column mapping."""
        table_name_lower = table_name.lower()

        # Check if table exists
        table_exists = True
        try:
            spark.read.table(table_name_lower)
            print(f"Table {table_name_lower} exists.")
        except Exception as e:
            print(f"Table {table_name_lower} does not exist.")
            table_exists = False

        # Columns to deduplicate on (excluding metadata)
        dedup_cols = [col for col in dataframe.columns if col not in ["Timestamp", "ExportName", "ExportDate"]]

        if table_exists:
            try:
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))

                existing_df = spark.read.table(table_name_lower)

                distinct_existing_df = existing_df.dropDuplicates(subset=dedup_cols)
                initial_existing_count = distinct_existing_df.count()
                print(f"Existing distinct count: {initial_existing_count}")

                distinct_new_df = dataframe.dropDuplicates(subset=dedup_cols)

                combined_distinct_df = distinct_new_df.unionByName(distinct_existing_df) \
                                                    .dropDuplicates(subset=dedup_cols)
                final_distinct_count = combined_distinct_df.count()
                print(f"Final distinct count: {final_distinct_count}")

                rows_added = final_distinct_count - initial_existing_count
                print(f"Added {rows_added} new distinct records to {table_name_lower}.")

                combined_distinct_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

            except Exception as e:
                print(f"Exception: Saving all records in new table. Exception: {e}")
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

        else:
            try:
                print(f"Table doesn't exist. Saving all records in new table.")
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")
            except Exception as e:
                print(f"Error saving data: {e}")

    table_prefix = 'BondedWarehouseRecordTracking_'

    record_tracking_df_to_table(combined_df, f"{table_prefix}shipments", file_name)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 51, Finished, Available, Finished)

# Init Send To Azure Blob Storage

In [51]:
if ready_to_copy == False:
    output_msg = f'Process Complete'

    notebookutils.notebook.exit(output_msg)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 52, Finished, Available, Finished)

## Copy the file to an ADLS account for loading to the SFTP
Set the source and destination paths

In [52]:
if "DEV" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_dev/ToBeSent/" + file_name
    
elif "UAT" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_uat/ToBeSent/" + file_name

else:
    dest_abfss_file_path = "Files/bonded_warehouse/ToBeSent/" + file_name

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 53, Finished, Available, Finished)

In [53]:
source_abfss_file_path = 'Files/Output/' + file_name

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 54, Finished, Available, Finished)

In [54]:
if transfer_file:
    notebookutils.fs.fastcp(source_abfss_file_path, dest_abfss_file_path)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 55, Finished, Available, Finished)

In [55]:
if ready_to_copy == True:
    output_msg = f'Process Complete'

notebookutils.notebook.exit(output_msg)

StatementMeta(, 160457bf-61f2-4e7e-b79c-f05a49e775f8, 56, Finished, Available, Finished)

ExitValue: Process Complete